# 05 Stratified Analysis and Confounding — Reference Solutions

Complete solutions to the stratified analysis exercises for the Pine and Cypress Nursing Home Legionnaires' disease cluster.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## Question 1: Confounding Analysis of Hydrotherapy Use

In [ ]:
# --- Crude RR ---
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a0 = int(ct_hydro.loc[1, 1])
b0 = int(ct_hydro.loc[1, 0])
c0 = int(ct_hydro.loc[0, 1])
d0 = int(ct_hydro.loc[0, 0])
crude_rr_hydro = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (hydrotherapy -> infected) = {crude_rr_hydro:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Functional status x Hydrotherapy use rate ===")
print(pd.crosstab(df["functional_status"], df["hydrotherapy_use"],
                  normalize="index").round(3))

# --- Stratum-specific RR ---
strata = sorted(df["functional_status"].unique())
hydro_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["hydrotherapy_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {s}: skipped")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hydro_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hydro_df = pd.DataFrame(hydro_results)
print("\n=== Stratum-specific RR ===")
for _, row in hydro_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  Crude RR = {crude_rr_hydro:.3f}")

## Question 2: Mantel-Haenszel Adjustment

In [ ]:
numerator = 0
denominator = 0

for _, row in hydro_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hydro = numerator / denominator

print(f"Mantel-Haenszel adjusted RR = {rr_mh_hydro:.3f}")
print(f"Crude RR                    = {crude_rr_hydro:.3f}")
print(f"Difference                  = {crude_rr_hydro - rr_mh_hydro:.3f}")

if abs(crude_rr_hydro - rr_mh_hydro) > 0.1:
    print("\n-> Functional status is indeed a confounder of hydrotherapy use (the crude RR was inflated)")
else:
    print("\n-> After controlling for functional status the RR barely changed; confounding effect is limited")

## Question 3 (Challenge): Stratify by Age Group + Forest Plot

In [ ]:
# Crude RR
ct_shower = pd.crosstab(df["shower_use"], df["infected"])
a_crude = int(ct_shower.loc[1, 1])
b_crude = int(ct_shower.loc[1, 0])
c_crude = int(ct_shower.loc[0, 1])
d_crude = int(ct_shower.loc[0, 0])
crude_rr = risk_ratio(a_crude, a_crude + b_crude, c_crude, c_crude + d_crude)

# Create age groups
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# Stratum-specific RR
age_results = []
for grp in ["60-69", "70-79", "80-89", "90+"]:
    sub = df[df["age_group"] == grp]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {grp}: skipped")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    age_results.append({
        "stratum": grp, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

age_df = pd.DataFrame(age_results)
print("=== Stratum-specific RR by age group ===")
for _, row in age_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(age_df))

ax.errorbar(
    age_df["RR"], y_pos,
    xerr=[age_df["RR"] - age_df["CI_lower"],
          age_df["CI_upper"] - age_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"Crude RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(age_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("Forest plot: shower use -> infection (stratified by age group)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MH adjusted RR
num = 0
den = 0
for _, row in age_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    num += a_i * (c_i + d_i) / n_i
    den += c_i * (a_i + b_i) / n_i

rr_mh_age = num / den

print(f"MH adjusted RR (controlling for age) = {rr_mh_age:.3f}")
print(f"Crude RR                             = {crude_rr:.3f}")
print(f"Difference                           = {crude_rr - rr_mh_age:.3f}")

# Homogeneity
rr_vals = age_df["RR"].values
print(f"\nRange of stratum-specific RRs: {rr_vals.min():.3f} – {rr_vals.max():.3f}")
if rr_vals.max() - rr_vals.min() > 0.5:
    print("-> The RRs across age groups differ substantially; age effect modification may be present")
else:
    print("-> The RRs across age groups are similar; age interaction is not notable")

### Interpretation

- **Functional status**: bedridden residents don't shower and are also less often infected; ambulatory residents shower more and are also more often infected → classic confounding
- **After MH adjustment**: if RR_MH is clearly smaller than the crude RR, functional status is confirmed as a confounder
- **Age stratification**: if the RRs across age groups are close, age interaction is small
- **Limitation**: stratified analysis can control for only one variable at a time → we need Ch06's logistic regression to adjust for multiple factors simultaneously

## Question 4 Solution

In [ ]:
# --- Data: age and comorbidity of COVID-19 cases ---
rng = np.random.default_rng(20)
n = 800

age_group = rng.choice(["under60", "60plus"], size=n, p=[0.6, 0.4])
comorbidity = np.array([
    rng.binomial(1, 0.5 if ag == "60plus" else 0.15) for ag in age_group
])
death_prob = np.select(
    [
        (age_group == "under60") & (comorbidity == 0),
        (age_group == "under60") & (comorbidity == 1),
        (age_group == "60plus") & (comorbidity == 0),
        (age_group == "60plus") & (comorbidity == 1),
    ],
    [0.02, 0.05, 0.12, 0.30],
)
death = rng.binomial(1, death_prob)

covid_df = pd.DataFrame({
    "case_id": [f"C{i:04d}" for i in range(n)],
    "age_group": age_group,
    "comorbidity": comorbidity,
    "death": death,
})

# --- Crude RR ---
ct_covid = pd.crosstab(covid_df["comorbidity"], covid_df["death"])
a0 = int(ct_covid.loc[1, 1])
b0 = int(ct_covid.loc[1, 0])
c0 = int(ct_covid.loc[0, 1])
d0 = int(ct_covid.loc[0, 0])
crude_rr_covid = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (comorbidity -> death) = {crude_rr_covid:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Age group x Comorbidity rate ===")
print(pd.crosstab(covid_df["age_group"], covid_df["comorbidity"], normalize="index").round(3))
print("\n=== Age group x Death rate ===")
print(pd.crosstab(covid_df["age_group"], covid_df["death"], normalize="index").round(3))

# --- Stratum-specific RR ---
covid_results = []
for ag in ["under60", "60plus"]:
    sub = covid_df[covid_df["age_group"] == ag]
    ct_s = pd.crosstab(sub["comorbidity"], sub["death"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    covid_results.append({
        "stratum": ag, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

covid_strat_df = pd.DataFrame(covid_results)
print("\n=== Stratum-specific RR by age group ===")
for _, row in covid_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  Crude RR = {crude_rr_covid:.3f}")

# --- Mantel-Haenszel adjusted RR ---
numerator = 0
denominator = 0
for _, row in covid_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_covid = numerator / denominator
print(f"\nMantel-Haenszel adjusted RR = {rr_mh_covid:.3f}")
print(f"Crude RR                    = {crude_rr_covid:.3f}")
print(f"Difference                  = {crude_rr_covid - rr_mh_covid:.3f}")

if crude_rr_covid - rr_mh_covid > 0.5:
    print("\n-> Age is a confounder of comorbidity's effect on mortality: older residents have both a higher comorbidity rate and a higher mortality risk, inflating the crude RR")
else:
    print("\n-> After controlling for age the RR barely changed; confounding effect is limited")

## Question 5 Solution

In [ ]:
# --- Data: influenza vaccination and age ---
rng = np.random.default_rng(11)
n = 800

age_group = rng.choice(["under65", "65plus"], size=n, p=[0.7, 0.3])
vaccinated = np.array([
    rng.binomial(1, 0.7 if ag == "65plus" else 0.3) for ag in age_group
])
infect_prob = np.select(
    [
        (age_group == "under65") & (vaccinated == 0),
        (age_group == "under65") & (vaccinated == 1),
        (age_group == "65plus") & (vaccinated == 0),
        (age_group == "65plus") & (vaccinated == 1),
    ],
    [0.20, 0.10, 0.40, 0.20],
)
infected = rng.binomial(1, infect_prob)

flu_df = pd.DataFrame({
    "case_id": [f"F{i:04d}" for i in range(n)],
    "age_group": age_group,
    "vaccinated": vaccinated,
    "infected": infected,
})

# --- Crude RR ---
ct_flu = pd.crosstab(flu_df["vaccinated"], flu_df["infected"])
a0 = int(ct_flu.loc[1, 1])
b0 = int(ct_flu.loc[1, 0])
c0 = int(ct_flu.loc[0, 1])
d0 = int(ct_flu.loc[0, 0])
crude_rr_flu = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (vaccinated -> infected) = {crude_rr_flu:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Age group x Vaccination rate ===")
print(pd.crosstab(flu_df["age_group"], flu_df["vaccinated"], normalize="index").round(3))
print("\n=== Age group x Infection rate ===")
print(pd.crosstab(flu_df["age_group"], flu_df["infected"], normalize="index").round(3))

# --- Stratum-specific RR ---
flu_results = []
for ag in ["under65", "65plus"]:
    sub = flu_df[flu_df["age_group"] == ag]
    ct_s = pd.crosstab(sub["vaccinated"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    flu_results.append({
        "stratum": ag, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

flu_strat_df = pd.DataFrame(flu_results)
print("\n=== Stratum-specific RR by age group ===")
for _, row in flu_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  Crude RR = {crude_rr_flu:.3f}")

# --- Mantel-Haenszel adjusted RR ---
numerator = 0
denominator = 0
for _, row in flu_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_flu = numerator / denominator
print(f"\nMantel-Haenszel adjusted RR = {rr_mh_flu:.3f}")
print(f"Crude RR                    = {crude_rr_flu:.3f}")

print("\n-> The crude RR is closer to 1 than the MH RR because age simultaneously raises both vaccination rate and infection risk (confounding by indication),")
print("  which masks the vaccine's protective effect; the MH-adjusted RR more accurately reflects the vaccine's true protective effect (a smaller, more protective RR).")

## Question 6 Solution

In [ ]:
# --- Data: hepatitis A cluster at a group meal ---
rng = np.random.default_rng(5)
n = 900

vaccinated = rng.binomial(1, 0.35, size=n)
ate_shellfish = np.array([
    rng.binomial(1, 0.3 if v == 1 else 0.6) for v in vaccinated
])
infect_prob = np.select(
    [
        (vaccinated == 0) & (ate_shellfish == 0),
        (vaccinated == 0) & (ate_shellfish == 1),
        (vaccinated == 1) & (ate_shellfish == 0),
        (vaccinated == 1) & (ate_shellfish == 1),
    ],
    [0.05, 0.35, 0.01, 0.07],
)
infected = rng.binomial(1, infect_prob)

hav_df = pd.DataFrame({
    "case_id": [f"H{i:04d}" for i in range(n)],
    "vaccinated": vaccinated,
    "ate_shellfish": ate_shellfish,
    "infected": infected,
})

# --- Crude RR ---
ct_hav = pd.crosstab(hav_df["ate_shellfish"], hav_df["infected"])
a0 = int(ct_hav.loc[1, 1])
b0 = int(ct_hav.loc[1, 0])
c0 = int(ct_hav.loc[0, 1])
d0 = int(ct_hav.loc[0, 0])
crude_rr_hav = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (ate_shellfish -> infected) = {crude_rr_hav:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Vaccination history x Shellfish consumption rate ===")
print(pd.crosstab(hav_df["vaccinated"], hav_df["ate_shellfish"], normalize="index").round(3))
print("\n=== Vaccination history x Infection rate ===")
print(pd.crosstab(hav_df["vaccinated"], hav_df["infected"], normalize="index").round(3))

# --- Stratum-specific RR ---
hav_results = []
for v in [0, 1]:
    sub = hav_df[hav_df["vaccinated"] == v]
    ct_s = pd.crosstab(sub["ate_shellfish"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hav_results.append({
        "stratum": "vaccinated" if v == 1 else "unvaccinated", "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hav_strat_df = pd.DataFrame(hav_results)
print("\n=== Stratum-specific RR by vaccination history ===")
for _, row in hav_strat_df.iterrows():
    print(f"  {row['stratum']:12s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  Crude RR = {crude_rr_hav:.3f}")

# --- Mantel-Haenszel adjusted RR ---
numerator = 0
denominator = 0
for _, row in hav_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hav = numerator / denominator
print(f"\nMantel-Haenszel adjusted RR = {rr_mh_hav:.3f}")
print(f"Crude RR                    = {crude_rr_hav:.3f}")
print(f"Difference                  = {crude_rr_hav - rr_mh_hav:.3f}")

if crude_rr_hav - rr_mh_hav > 0.5:
    print("\n-> Vaccination history is a confounder: unvaccinated individuals ate shellfish more often and had a higher infection risk, inflating the crude RR")
else:
    print("\n-> After controlling for vaccination history the RR barely changed; confounding effect is limited")

## Question 7 Solution

In [ ]:
# --- Data: cross-regional dengue fever investigation ---
rng = np.random.default_rng(42)
n = 900

region = rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.35, 0.25])
water_p = {"urban": 0.2, "suburban": 0.4, "rural": 0.6}
standing_water = np.array([rng.binomial(1, water_p[r]) for r in region])
infect_p = {
    ("urban", 0): 0.03, ("urban", 1): 0.09,
    ("suburban", 0): 0.08, ("suburban", 1): 0.24,
    ("rural", 0): 0.15, ("rural", 1): 0.45,
}
infect_prob = np.array([infect_p[(r, w)] for r, w in zip(region, standing_water)])
infected = rng.binomial(1, infect_prob)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(n)],
    "region": region,
    "standing_water": standing_water,
    "infected": infected,
})

# --- Crude RR ---
ct_dengue = pd.crosstab(dengue_df["standing_water"], dengue_df["infected"])
a0 = int(ct_dengue.loc[1, 1])
b0 = int(ct_dengue.loc[1, 0])
c0 = int(ct_dengue.loc[0, 1])
d0 = int(ct_dengue.loc[0, 0])
crude_rr_dengue = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (standing_water -> infected) = {crude_rr_dengue:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Region x Standing water rate ===")
print(pd.crosstab(dengue_df["region"], dengue_df["standing_water"], normalize="index").round(3))
print("\n=== Region x Infection rate ===")
print(pd.crosstab(dengue_df["region"], dengue_df["infected"], normalize="index").round(3))

# --- Stratum-specific RR ---
dengue_results = []
for r_ in ["urban", "suburban", "rural"]:
    sub = dengue_df[dengue_df["region"] == r_]
    ct_s = pd.crosstab(sub["standing_water"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    dengue_results.append({
        "stratum": r_, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

dengue_strat_df = pd.DataFrame(dengue_results)
print("\n=== Stratum-specific RR by region ===")
for _, row in dengue_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  Crude RR = {crude_rr_dengue:.3f}")

# --- Mantel-Haenszel adjusted RR ---
numerator = 0
denominator = 0
for _, row in dengue_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_dengue = numerator / denominator
print(f"\nMantel-Haenszel adjusted RR = {rr_mh_dengue:.3f}")
print(f"Crude RR                    = {crude_rr_dengue:.3f}")
print(f"Difference                  = {crude_rr_dengue - rr_mh_dengue:.3f}")

rr_vals = dengue_strat_df["RR"].values
print(f"\nRange of stratum-specific RRs by region: {rr_vals.min():.3f} – {rr_vals.max():.3f}")
if crude_rr_dengue - rr_mh_dengue > 0.5:
    print("-> Region is a confounder of the standing water exposure and dengue infection association: rural areas have both a higher standing-water rate and a higher infection risk, inflating the crude RR")
else:
    print("-> After controlling for region the RR barely changed; confounding effect is limited")

## Question 8 Solution

In [ ]:
# --- Data: tuberculosis contact screening ---
from epi_learning.metrics import odds_ratio

rng = np.random.default_rng(291)
n = 900

diabetes = rng.binomial(1, 0.25, size=n)
close_contact = np.array([
    rng.binomial(1, 0.5 if d == 1 else 0.25) for d in diabetes
])
tb_prob = np.select(
    [
        (diabetes == 0) & (close_contact == 0),
        (diabetes == 0) & (close_contact == 1),
        (diabetes == 1) & (close_contact == 0),
        (diabetes == 1) & (close_contact == 1),
    ],
    [0.02, 0.10, 0.08, 0.32],
)
active_tb = rng.binomial(1, tb_prob)

tb_df = pd.DataFrame({
    "case_id": [f"T{i:04d}" for i in range(n)],
    "diabetes": diabetes,
    "close_contact": close_contact,
    "active_tb": active_tb,
})

# --- Crude OR ---
ct_tb = pd.crosstab(tb_df["close_contact"], tb_df["active_tb"])
a0 = int(ct_tb.loc[1, 1])
b0 = int(ct_tb.loc[1, 0])
c0 = int(ct_tb.loc[0, 1])
d0 = int(ct_tb.loc[0, 0])
crude_or_tb = odds_ratio(a0, b0, c0, d0)
print(f"Crude OR (close_contact -> active_tb) = {crude_or_tb:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Diabetes x Close contact rate ===")
print(pd.crosstab(tb_df["diabetes"], tb_df["close_contact"], normalize="index").round(3))
print("\n=== Diabetes x Active TB rate ===")
print(pd.crosstab(tb_df["diabetes"], tb_df["active_tb"], normalize="index").round(3))

# --- Stratum-specific OR ---
tb_results = []
for d_ in [0, 1]:
    sub = tb_df[tb_df["diabetes"] == d_]
    ct_s = pd.crosstab(sub["close_contact"], sub["active_tb"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    or_s = odds_ratio(a_s, b_s, c_s, d_s)
    ln_or = np.log(or_s)
    se = np.sqrt(1/a_s + 1/b_s + 1/c_s + 1/d_s)
    ci_lo = np.exp(ln_or - 1.96 * se)
    ci_hi = np.exp(ln_or + 1.96 * se)
    tb_results.append({
        "stratum": "diabetic" if d_ == 1 else "non_diabetic", "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "OR": or_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

tb_strat_df = pd.DataFrame(tb_results)
print("\n=== Stratum-specific OR by diabetes status ===")
for _, row in tb_strat_df.iterrows():
    print(f"  {row['stratum']:14s}  OR={row['OR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  Crude OR = {crude_or_tb:.3f}")

# --- Forest plot ---
fig, ax = plt.subplots(figsize=(8, 3.5))
y_pos = range(len(tb_strat_df))

ax.errorbar(
    tb_strat_df["OR"], y_pos,
    xerr=[tb_strat_df["OR"] - tb_strat_df["CI_lower"],
          tb_strat_df["CI_upper"] - tb_strat_df["OR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_or_tb, color="red", linestyle=":", alpha=0.7,
           label=f"Crude OR={crude_or_tb:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(tb_strat_df["stratum"])
ax.set_xlabel("Odds Ratio (OR)")
ax.set_title("Forest plot: close contact -> active tuberculosis (stratified by diabetes status)")
ax.legend()
plt.tight_layout()
plt.show()

# --- Mantel-Haenszel adjusted OR ---
numerator = 0
denominator = 0
for _, row in tb_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * d_i / n_i
    denominator += b_i * c_i / n_i

or_mh_tb = numerator / denominator
print(f"\nMantel-Haenszel adjusted OR = {or_mh_tb:.3f}")
print(f"Crude OR                    = {crude_or_tb:.3f}")
print(f"Difference                  = {crude_or_tb - or_mh_tb:.3f}")

or_vals = tb_strat_df["OR"].values
print(f"\nRange of stratum-specific ORs: {or_vals.min():.3f} – {or_vals.max():.3f}")
if or_vals.max() - or_vals.min() > 1.5:
    print("-> The stratum-specific ORs differ substantially; diabetes may be an effect modifier")
else:
    print("-> The stratum-specific ORs are similar; diabetes is more likely a pure confounder rather than an effect modifier")

if crude_or_tb - or_mh_tb > 1.0:
    print("-> Diabetes meets the confounder conditions: diabetic patients have both a higher close-contact rate and higher TB risk, inflating the crude OR")
else:
    print("-> After controlling for diabetes the OR barely changed; confounding effect is limited")